# 12 — Origin Classification: 5-Way Regional Grouping

Reframes the origin-classification task from country-level (15 classes) to region-level.
The country-level ceiling on scrubbed+ full-concat text is ~0.64 macro-F1 (07.1, 09). Regional
grouping asks a different but thesis-worthy question: *can the model identify the broad
producing region when direct country names are scrubbed?* The hypothesis is that flavor-profile
signal (floral-citric East African, balanced-nutty Central American, earthy Indonesian) survives
the scrub even when Ethiopia-vs-Kenya distinctions do not.

Regions (5-way — splits USA from Asia-Pacific):
- **East Africa** — Ethiopia, Kenya, Rwanda, Burundi, Tanzania, Uganda, DR Congo, Zambia, Zimbabwe, Cameroon, Malawi, South Africa, Yemen
- **Central America** — Guatemala, Costa Rica, Panama, El Salvador, Honduras, Nicaragua, Mexico, Jamaica, Puerto Rico, Haiti, Dominican Republic
- **South America** — Colombia, Peru, Brazil, Ecuador, Bolivia, Venezuela
- **Asia-Pacific** — Indonesia, Taiwan, Thailand, PNG, Philippines, India, Vietnam, China, Timor-Leste, Malaysia, Laos, Nepal, Myanmar, Australia, UK
- **USA** — United States (Hawaii / California)

Expected: ~7,585 rows across 5 classes. Expected macro-F1: 0.78–0.88.

- Input: `text_full_concat_scrubbed_plus` (Blind Assessment + Notes + Who Should Drink It + Bottom Line, 4-tier scrubbed)
- Models: RoBERTa-base (weighted CE, lr=2e-5) and ModernBERT-base (plain CE, lr=3e-5)
- Seeds: [42, 123, 2024]


In [1]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    balanced_accuracy_score, f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments,
)
from transformers.utils.notebook import NotebookProgressCallback

RANDOM_STATE = 42
SEEDS = [42, 123, 2024]
TEXT_COLUMN = 'text_full_concat_scrubbed_plus'
MAX_LENGTH = 256

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
USE_FP16 = True

ROBERTA_CKPT = 'roberta-base'
ROBERTA_BEST_LR = 2e-5
ROBERTA_BEST_WEIGHTED = True

MODERNBERT_CKPT = 'answerdotai/ModernBERT-base'
MODERNBERT_BEST_LR = 3e-5
MODERNBERT_BEST_WEIGHTED = False

OUTPUT_DIR_ROOT = 'artifacts/origin_region_5way_scrubbed_plus'
os.makedirs(OUTPUT_DIR_ROOT, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

COUNTRY_TO_REGION = {
    # --- East Africa ---
    'Ethiopia': 'East Africa',
    'Kenya': 'East Africa',
    'Rwanda': 'East Africa',
    'Burundi': 'East Africa',
    'Tanzania': 'East Africa',
    'Uganda': 'East Africa',
    'DR Congo': 'East Africa',
    'Zambia': 'East Africa',
    'Zimbabwe': 'East Africa',
    'Cameroon': 'East Africa',
    'Malawi': 'East Africa',
    'South Africa': 'East Africa',
    'Yemen': 'East Africa',
    # --- Central America ---
    'Guatemala': 'Central America',
    'Costa Rica': 'Central America',
    'Panama': 'Central America',
    'El Salvador': 'Central America',
    'Honduras': 'Central America',
    'Nicaragua': 'Central America',
    'Mexico': 'Central America',
    'Jamaica': 'Central America',
    'Puerto Rico': 'Central America',
    'Haiti': 'Central America',
    'Dominican Republic': 'Central America',
    # --- South America ---
    'Colombia': 'South America',
    'Peru': 'South America',
    'Brazil': 'South America',
    'Ecuador': 'South America',
    'Bolivia': 'South America',
    'Venezuela': 'South America',
    # --- Asia-Pacific ---
    'Indonesia': 'Asia-Pacific',
    'Taiwan': 'Asia-Pacific',
    'Thailand': 'Asia-Pacific',
    'Papua New Guinea': 'Asia-Pacific',
    'Philippines': 'Asia-Pacific',
    'India': 'Asia-Pacific',
    'Vietnam': 'Asia-Pacific',
    'China': 'Asia-Pacific',
    'Timor-Leste': 'Asia-Pacific',
    'Malaysia': 'Asia-Pacific',
    'Laos': 'Asia-Pacific',
    'Nepal': 'Asia-Pacific',
    'Myanmar': 'Asia-Pacific',
    'Australia': 'Asia-Pacific',
    'United Kingdom': 'Asia-Pacific',
    # --- USA ---
    'United States': 'USA',
}


Device: cuda


## Scrubbing vocabulary (identical to 07.1 / 04.5)

In [2]:
# Scrubbing vocabulary: countries + region aliases + cultivars + producer context
COUNTRY_TERMS = {
    "Ethiopia": ["ethiopia", "ethiopian"],
    "Colombia": ["colombia", "colombian", "columbian"],
    "Panama": ["panama", "panamanian"],
    "Kenya": ["kenya", "kenyan"],
    "Indonesia": ["indonesia", "indonesian"],
    "Guatemala": ["guatemala", "guatemalan"],
    "Costa Rica": ["costa rica", "costa rican", "costarican"],
    "El Salvador": ["el salvador", "salvadoran", "salvadorean", "salvadorian"],
    "Rwanda": ["rwanda", "rwandan"],
    "Brazil": ["brazil", "brazilian"],
    "Honduras": ["honduras", "honduran"],
    "Peru": ["peru", "peruvian"],
    "Taiwan": ["taiwan", "taiwanese"],
    "Papua New Guinea": ["papua new guinea", "papua new guinean"],
    "Nicaragua": ["nicaragua", "nicaraguan"],
    "Burundi": ["burundi", "burundian"],
    "Thailand": ["thailand", "thai"],
    "India": ["india", "indian"],
    "Mexico": ["mexico", "mexican"],
    "Tanzania": ["tanzania", "tanzanian"],
    "Yemen": ["yemen", "yemeni"],
    "Ecuador": ["ecuador", "ecuadorian"],
    "Bolivia": ["bolivia", "bolivian"],
    "Jamaica": ["jamaica", "jamaican"],
    "Uganda": ["uganda", "ugandan"],
    "Dominican Republic": ["dominican republic", "dominican"],
    "Zambia": ["zambia", "zambian"],
    "China": ["china", "chinese"],
    "Vietnam": ["vietnam", "vietnamese"],
    "Philippines": ["philippines", "philippine", "filipino"],
    "Malaysia": ["malaysia", "malaysian"],
    "Laos": ["laos", "laotian"],
    "Zimbabwe": ["zimbabwe", "zimbabwean"],
    "Haiti": ["haiti", "haitian"],
    "Puerto Rico": ["puerto rico", "puerto rican", "puertorican"],
    "Nepal": ["nepal", "nepalese", "nepali"],
    "Myanmar": ["myanmar", "burmese"],
    "Cameroon": ["cameroon", "cameroonian"],
    "Australia": ["australia", "australian"],
    "South Africa": ["south africa", "south african"],
    "Malawi": ["malawi", "malawian"],
    "Venezuela": ["venezuela", "venezuelan", "merida state", "mocoties valley"],
    "Timor-Leste": ["east timor", "timor leste", "timorese"],
    "DR Congo": ["democratic republic of the congo", "dr congo", "congo", "congolese"],
    "United Kingdom": ["united kingdom", "british", "pitcairn island", "saint helena", "st helena", "sandy bay valley"],
    "United States": [
        "usa", "united states", "american",
        "hawaii", "hawai'i", "hawaiian", "hawaii island",
        "big island", "kona", "puna district", "holualoa", "oahu", "maui", "kauai",
        "ka u", "ka'u", "kau",
    ],
}

REGION_ALIASES = {
    "Ethiopia": ["yirgacheffe", "sidamo", "sidama", "guji", "gedeb", "gedeo", "kochere", "hambela",
                 "shakiso", "jimma", "limu", "oromia", "harrar", "kaffa", "bench maji", "bench-maji",
                 "arbegona", "bensa"],
    "Colombia": ["huila", "cauca", "narino", "tolima", "quindio", "caldas", "risaralda", "antioquia",
                 "cundinamarca", "santander", "pitalito", "acevedo", "planadas", "gaitania", "piendamo",
                 "caicedonia", "san agustin", "armenia"],
    "Panama": ["boquete", "chiriqui", "volcan", "jaramillo", "alto quiel", "paso ancho",
               "piedra candela", "canas verdes", "silla del pando", "renacimiento"],
    "Kenya": ["nyeri", "kirinyaga", "kiambu", "embu", "muranga", "murang'a", "thika", "ruiru",
              "mathira", "meru", "nakuru", "gichugu", "karatina"],
    "Indonesia": ["sumatra", "aceh", "gayo", "lintong", "mandheling", "sidikalang", "toraja",
                  "sulawesi", "java", "bali", "kintamani", "flores", "kerinci"],
    "Guatemala": ["huehuetenango", "antigua", "acatenango", "fraijanes", "coban", "atitlan",
                  "chimaltenango", "quiche", "solola", "palencia", "sacatepequez", "san marcos",
                  "hoja blanca", "cuilco"],
    "Costa Rica": ["tarrazu", "central valley", "west valley", "tres rios", "naranjo", "dota",
                   "poas", "alajuela", "brunca", "turrialba", "coto brus", "chirripo"],
    "El Salvador": ["ahuachapan", "chalatenango", "apaneca", "ilamatepec", "santa ana",
                    "el boqueron", "quetzaltepec", "juayua", "ataco"],
    "Rwanda": ["gakenke", "nyamasheke", "karongi", "huye", "nyamagabe", "rulindo", "gikongoro",
               "lake kivu"],
    "Brazil": ["cerrado", "mogiana", "minas gerais", "mantiqueira", "sul de minas",
               "chapada diamantina", "carmo de minas"],
    "Honduras": ["marcala", "copan", "ocotepeque", "comayagua", "intibuca", "santa barbara",
                 "el paraiso", "capucas"],
    "Peru": ["cajamarca", "jaen", "san ignacio", "chanchamayo", "cusco", "villa rica", "oxapampa",
             "junin", "puno"],
    "Taiwan": ["alishan", "yunlin", "chiayi", "chia yi", "nantou", "taichung", "pingtung"],
    "Papua New Guinea": ["wahgi valley", "western highlands", "eastern highlands", "jiwaka",
                         "kainantu", "okapa"],
    "Nicaragua": ["jinotega", "matagalpa", "nueva segovia", "madriz", "ocotal", "dipilto"],
    "Burundi": ["kayanza", "ngozi", "muramvya", "muyinga", "bururi"],
    "Thailand": ["chiang rai", "doi chang", "doi pangkhon", "chiang mai", "nan province"],
    "India": ["coorg", "chikmagalur", "karnataka", "bababudangiri", "nilgiris"],
    "Mexico": ["chiapas", "oaxaca", "veracruz", "coatepec", "pluma hidalgo"],
    "Tanzania": ["mbeya", "ruvuma", "ngorongoro", "arusha", "kilimanjaro", "mbozi"],
    "Yemen": ["haraaz", "haraz", "sanaa", "bani matar", "hayma"],
    "Ecuador": ["loja", "pichincha", "imbabura", "zamora", "chimborazo", "saraguro"],
    "Bolivia": ["caranavi", "yungas", "la paz"],
    "Jamaica": ["blue mountain", "blue mountains"],
    "Uganda": ["rwenzori", "bugisu", "sipi falls"],
    "Vietnam": ["lam dong", "quang tri", "dalat", "cau dat"],
    "Philippines": ["benguet", "bukidnon", "davao"],
    "China": ["yunnan", "baoshan", "puer", "pu'er", "lincong"],
    "Puerto Rico": ["utuado", "yauco", "adjuntas"],
    "Venezuela": ["mocoties valley", "merida state"],
    "Malawi": ["malawi"],
}

REGION_ONLY_TERMS = [
    "central america", "south america", "latin america",
    "east africa", "central africa", "west africa", "africa",
    "asia", "the americas", "americas",
    "central and south america", "south and central america",
    "east and central africa", "various africa growing regions",
]

CULTIVAR_TERMS = [
    "sl28", "sl-28", "sl 28", "sl34", "sl-34", "sl 34",
    "ruiru 11", "ruiru-11", "batian", "k7",
    "gesha", "geisha",
    "pacamara", "pache", "villa sarchi",
    "bourbon", "pink bourbon", "yellow bourbon", "red bourbon",
    "caturra", "catuai", "catuai vermelho", "catuai amarelo",
    "typica", "mundo novo",
    "maragogipe", "maragogype", "maracaturra",
    "castillo", "colombia variety", "variedad colombia", "tabi",
    "heirloom", "ethiopian heirloom",
    "tim tim", "s795", "ateng",
    "catimor", "sarchimor", "icatu", "obata",
]

PRODUCER_CONTEXT_TERMS = [
    "finca", "hacienda", "beneficio",
    "cup of excellence",
    "washing station", "wet mill",
    "best of panama",
]

all_scrub_terms = []
for v in COUNTRY_TERMS.values():    all_scrub_terms.extend(v)
for v in REGION_ALIASES.values():   all_scrub_terms.extend(v)
all_scrub_terms.extend(REGION_ONLY_TERMS)
all_scrub_terms.extend(CULTIVAR_TERMS)
all_scrub_terms.extend(PRODUCER_CONTEXT_TERMS)
all_scrub_terms = sorted(set(all_scrub_terms), key=len, reverse=True)
print(f'Loaded {len(all_scrub_terms)} scrub terms')

SCRUB_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(t) for t in all_scrub_terms) + r')\b',
    flags=re.IGNORECASE,
)

def scrub(text):
    if not isinstance(text, str) or not text:
        return ''
    return SCRUB_PATTERN.sub(' [ORIGIN] ', text)

Loaded 403 scrub terms


## Build text column, map country -> region, split


In [3]:
def minimal_raw_text(text):
    text = '' if pd.isna(text) else str(text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def normalize_text_keep_brackets(text):
    text = minimal_raw_text(text)
    text = re.sub(r'[^a-z0-9\s\[\]]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

EXTRA_TEXT_COLS = ['Blind Assessment', 'Notes', 'Who Should Drink It', 'Bottom Line']

def concat_and_scrub(row):
    pieces = []
    for col in EXTRA_TEXT_COLS:
        v = row[col]
        if pd.notna(v) and str(v).strip():
            raw = minimal_raw_text(v)
            scrubbed = scrub(raw)
            cleaned = normalize_text_keep_brackets(scrubbed)
            pieces.append(cleaned)
    return ' '.join(p for p in pieces if p)

df = pd.read_csv('Data/final_coffee_reviews.csv')
df['text_full_concat_scrubbed_plus'] = df.apply(concat_and_scrub, axis=1)
df['text_raw_minimal'] = df['Blind Assessment'].fillna('').map(minimal_raw_text)
df['origin_country'] = df['Country'].astype('string').str.strip().replace('', pd.NA)
df['origin_region'] = df['origin_country'].map(COUNTRY_TO_REGION)

# Filters: text length >=30, origin country present, country maps to a region
work = df[
    (df['text_raw_minimal'].str.len() >= 30)
    & (df['origin_country'].notna())
    & (df['origin_region'].notna())
].copy().reset_index(drop=True)

# Leakage audit — now we check country leakage, since scrubbing was built for country names
def contains_own_country(row):
    t = row['text_full_concat_scrubbed_plus'].lower()
    c = str(row['origin_country']).lower()
    return c in t if c and t else False
work['leaks_country'] = work.apply(contains_own_country, axis=1)
leak_rate = float(work['leaks_country'].mean())

# Also audit region-name leakage (looser substring match)
def contains_region_token(row):
    t = row['text_full_concat_scrubbed_plus'].lower()
    reg = str(row['origin_region']).lower()
    tokens = [tok for tok in reg.replace('-', ' ').split() if len(tok) >= 4]
    return any(tok in t for tok in tokens)
work['leaks_region'] = work.apply(contains_region_token, axis=1)
region_leak_rate = float(work['leaks_region'].mean())

print(f'Task: 5-way regional classification')
print(f'Rows: {len(work)} | Regions: {work["origin_region"].nunique()}')
print(f'Avg scrubbed+ text length (chars): {int(work["text_full_concat_scrubbed_plus"].str.len().mean())}')
print(f'Country-name leakage: {leak_rate:.2%}')
print(f'Region-token leakage: {region_leak_rate:.2%}  (region words like "africa", "america" still in text)')
print()
print('Region distribution:')
print(work['origin_region'].value_counts())
print()
print('Country distribution (for reference):')
print(work['origin_country'].value_counts().head(25))


Task: 5-way regional classification
Rows: 7585 | Regions: 5
Avg scrubbed+ text length (chars): 876
Country-name leakage: 0.88%
Region-token leakage: 3.48%  (region words like "africa", "america" still in text)

Region distribution:
origin_region
East Africa        3141
Central America    1936
South America      1445
Asia-Pacific        761
USA                 302
Name: count, dtype: int64

Country distribution (for reference):
origin_country
Ethiopia            1996
Colombia             988
Kenya                699
Guatemala            561
Indonesia            452
Costa Rica           373
Panama               354
United States        302
El Salvador          240
Brazil               200
Rwanda               176
Peru                 130
Nicaragua            125
Honduras             123
Mexico               101
Burundi               85
Papua New Guinea      79
Ecuador               75
Taiwan                71
Tanzania              68
Thailand              68
Bolivia               51
Yeme

## Split + labels + class weights

In [4]:
y = work['origin_region']
X = work['text_full_concat_scrubbed_plus'].tolist()

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.15/0.85, stratify=y_tmp, random_state=RANDOM_STATE,
)

label_names = sorted(y.unique().tolist())
label2id = {n: i for i, n in enumerate(label_names)}
id2label = {i: n for i, n in enumerate(label_names)}

train_texts, val_texts, test_texts = X_train, X_val, X_test
train_labels = [label2id[y_] for y_ in y_train]
val_labels   = [label2id[y_] for y_ in y_val]
test_labels  = [label2id[y_] for y_ in y_test]

class_weights = compute_class_weight(class_weight='balanced', classes=np.arange(len(label_names)), y=train_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print(f'Train/Val/Test: {len(train_labels)} / {len(val_labels)} / {len(test_labels)}')
print(f'Labels ({len(label_names)}): {label_names}')
print(f'Class weights: {dict(zip(label_names, class_weights.round(3)))}')


Train/Val/Test: 5309 / 1138 / 1138
Labels (5): ['Asia-Pacific', 'Central America', 'East Africa', 'South America', 'USA']
Class weights: {'Asia-Pacific': 1.992, 'Central America': 0.784, 'East Africa': 0.483, 'South America': 1.05, 'USA': 5.008}


## Dataset + metrics + run_one

In [5]:
class CoffeeOriginDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(texts, truncation=True, max_length=max_length, padding=False)
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'balanced_accuracy': balanced_accuracy_score(labels, preds),
        'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1,
    }

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def run_one(model_checkpoint, learning_rate, use_class_weights, seed, epochs, early_stop_patience, tag):
    set_seed(seed)
    out_dir = os.path.join(OUTPUT_DIR_ROOT, tag)

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    train_ds = CoffeeOriginDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
    val_ds   = CoffeeOriginDataset(val_texts,   val_labels,   tokenizer, MAX_LENGTH)
    test_ds  = CoffeeOriginDataset(test_texts,  test_labels,  tokenizer, MAX_LENGTH)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint, num_labels=len(label_names), id2label=id2label, label2id=label2id,
    )
    args_kwargs = dict(
        output_dir=out_dir, learning_rate=learning_rate,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=epochs,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        eval_strategy='epoch', logging_strategy='epoch',
        disable_tqdm=True, report_to='none',
        fp16=USE_FP16 and torch.cuda.is_available(), seed=seed,
    )
    if early_stop_patience is not None:
        args_kwargs.update(dict(
            save_strategy='epoch', save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model='f1_macro', greater_is_better=True,
        ))
    else:
        args_kwargs.update(dict(save_strategy='no'))

    args = TrainingArguments(**args_kwargs)
    trainer_cls = WeightedTrainer if use_class_weights else Trainer
    extra = dict(class_weights=class_weights_tensor) if use_class_weights else {}
    callbacks = []
    if early_stop_patience is not None:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stop_patience))
    trainer = trainer_cls(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=compute_metrics, callbacks=callbacks, **extra,
    )
    try: trainer.remove_callback(NotebookProgressCallback)
    except Exception: pass
    print(f'\n=== {tag} | model={model_checkpoint} | lr={learning_rate} | weighted={use_class_weights} | ep={epochs} | seed={seed} ===')
    trainer.train()
    val_m  = trainer.evaluate(eval_dataset=val_ds)
    test_m = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='test')
    del model, trainer
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {
        'tag': tag, 'model': model_checkpoint,
        'lr': learning_rate, 'weighted': use_class_weights,
        'epochs': epochs, 'early_stop_patience': early_stop_patience, 'seed': seed,
        'val_f1_macro': val_m['eval_f1_macro'],
        'val_bal_acc': val_m['eval_balanced_accuracy'],
        'val_accuracy': val_m['eval_accuracy'],
        'test_f1_macro': test_m['test_f1_macro'],
        'test_bal_acc': test_m['test_balanced_accuracy'],
        'test_accuracy': test_m['test_accuracy'],
    }

## Part 1 — RoBERTa × 3 seeds

In [6]:
roberta_seed_results = []
for s in SEEDS:
    r = run_one(
        model_checkpoint=ROBERTA_CKPT,
        learning_rate=ROBERTA_BEST_LR,
        use_class_weights=ROBERTA_BEST_WEIGHTED,
        seed=s, epochs=12, early_stop_patience=3,
        tag=f'roberta_region5way_seed{s}',
    )
    roberta_seed_results.append(r)

roberta_df = pd.DataFrame(roberta_seed_results)[['seed','val_f1_macro','test_f1_macro','test_bal_acc','test_accuracy']]
print(roberta_df.round(4).to_string(index=False))
print()
print('Mean ± Std:')
print(roberta_df.drop(columns=['seed']).agg(['mean','std']).round(4).to_string())


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



=== roberta_region5way_seed42 | model=roberta-base | lr=2e-05 | weighted=True | ep=12 | seed=42 ===
{'loss': '3.113', 'grad_norm': '51.24', 'learning_rate': '1.955e-05', 'epoch': '1'}
{'eval_loss': '1.169', 'eval_accuracy': '0.5712', 'eval_balanced_accuracy': '0.5269', 'eval_precision_macro': '0.6252', 'eval_recall_macro': '0.5269', 'eval_f1_macro': '0.5224', 'eval_runtime': '1.73', 'eval_samples_per_second': '657.8', 'eval_steps_per_second': '20.81', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.819', 'grad_norm': '27.02', 'learning_rate': '1.779e-05', 'epoch': '2'}
{'eval_loss': '0.7383', 'eval_accuracy': '0.688', 'eval_balanced_accuracy': '0.7042', 'eval_precision_macro': '0.6301', 'eval_recall_macro': '0.7042', 'eval_f1_macro': '0.6527', 'eval_runtime': '1.698', 'eval_samples_per_second': '670.4', 'eval_steps_per_second': '21.21', 'epoch': '2'}


...

{'train_runtime': '364.5', 'train_samples_per_second': '174.8', 'train_steps_per_second': '5.465', 'train_loss': '0.9343', 'epoch': '10'}


There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': '0.6641', 'eval_accuracy': '0.7996', 'eval_balanced_accuracy': '0.7957', 'eval_precision_macro': '0.8093', 'eval_recall_macro': '0.7957', 'eval_f1_macro': '0.802', 'eval_runtime': '1.871', 'eval_samples_per_second': '608.3', 'eval_steps_per_second': '19.24', 'epoch': '10'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '0.6668', 'test_accuracy': '0.8014', 'test_balanced_accuracy': '0.7849', 'test_precision_macro': '0.7953', 'test_recall_macro': '0.7849', 'test_f1_macro': '0.7896', 'test_runtime': '1.659', 'test_samples_per_second': '685.9', 'test_steps_per_second': '21.7', 'epoch': '10'}
 seed  val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
   42        0.8103         0.8024        0.8022         0.8190
  123        0.7948         0.8078        0.8121         0.8225
 2024        0.8020         0.7896        0.7849         0.8014

Mean ± Std:
      val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
mean        0.8024         0.7999        0.7997         0.8143
std         0.0077         0.0094        0.0138         0.0113


## Part 2 — ModernBERT × 3 seeds

In [7]:
modernbert_seed_results = []
for s in SEEDS:
    r = run_one(
        model_checkpoint=MODERNBERT_CKPT,
        learning_rate=MODERNBERT_BEST_LR,
        use_class_weights=MODERNBERT_BEST_WEIGHTED,
        seed=s, epochs=12, early_stop_patience=3,
        tag=f'modernbert_region5way_seed{s}',
    )
    modernbert_seed_results.append(r)

modernbert_df = pd.DataFrame(modernbert_seed_results)[['seed','val_f1_macro','test_f1_macro','test_bal_acc','test_accuracy']]
print(modernbert_df.round(4).to_string(index=False))
print()
print('Mean ± Std:')
print(modernbert_df.drop(columns=['seed']).agg(['mean','std']).round(4).to_string())

print()
print('--- Head-to-head on 5-way regional scrubbed+ ---')
print(f'RoBERTa    (weighted, lr=2e-5): {roberta_df["test_f1_macro"].mean():.4f} ± {roberta_df["test_f1_macro"].std():.4f}')
print(f'ModernBERT (plain,    lr=3e-5): {modernbert_df["test_f1_macro"].mean():.4f} ± {modernbert_df["test_f1_macro"].std():.4f}')
print()
print('Reference — country-level (07.1, 15 classes, 6820 rows):')
print('  RoBERTa    0.6427 ± 0.0068')
print('  ModernBERT 0.5903 ± 0.0126')


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.



=== modernbert_region5way_seed42 | model=answerdotai/ModernBERT-base | lr=3e-05 | weighted=False | ep=12 | seed=42 ===
{'loss': '2.533', 'grad_norm': '15.41', 'learning_rate': '2.931e-05', 'epoch': '1'}
{'eval_loss': '0.9498', 'eval_accuracy': '0.5984', 'eval_balanced_accuracy': '0.407', 'eval_precision_macro': '0.647', 'eval_recall_macro': '0.407', 'eval_f1_macro': '0.3966', 'eval_runtime': '3.742', 'eval_samples_per_second': '304.1', 'eval_steps_per_second': '9.62', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '1.564', 'grad_norm': '20.57', 'learning_rate': '2.668e-05', 'epoch': '2'}
{'eval_loss': '0.703', 'eval_accuracy': '0.7311', 'eval_balanced_accuracy': '0.6552', 'eval_precision_macro': '0.7576', 'eval_recall_macro': '0.6552', 'eval_f1_macro': '0.6843', 'eval_runtime': '3.636', 'eval_samples_per_second': '313', 'eval_steps_per_second': '9.902', 'epoch': '2'}


...

{'loss': '0.0491', 'grad_norm': '18.47', 'learning_rate': '1.338e-05', 'epoch': '7'}
{'eval_loss': '1.504', 'eval_accuracy': '0.7821', 'eval_balanced_accuracy': '0.7241', 'eval_precision_macro': '0.8157', 'eval_recall_macro': '0.7241', 'eval_f1_macro': '0.7571', 'eval_runtime': '3.649', 'eval_samples_per_second': '311.9', 'eval_steps_per_second': '9.866', 'epoch': '7'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '402.3', 'train_samples_per_second': '158.4', 'train_steps_per_second': '4.952', 'train_loss': '0.916', 'epoch': '7'}
{'eval_loss': '0.6404', 'eval_accuracy': '0.7856', 'eval_balanced_accuracy': '0.7559', 'eval_precision_macro': '0.7885', 'eval_recall_macro': '0.7559', 'eval_f1_macro': '0.7704', 'eval_runtime': '3.822', 'eval_samples_per_second': '297.7', 'eval_steps_per_second': '9.418', 'epoch': '7'}


early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


{'test_loss': '0.6935', 'test_accuracy': '0.7636', 'test_balanced_accuracy': '0.7395', 'test_precision_macro': '0.7367', 'test_recall_macro': '0.7395', 'test_f1_macro': '0.7371', 'test_runtime': '3.61', 'test_samples_per_second': '315.3', 'test_steps_per_second': '9.973', 'epoch': '7'}
 seed  val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
   42        0.7933         0.7775        0.7657         0.7856
  123        0.7610         0.7319        0.7271         0.7531
 2024        0.7704         0.7371        0.7395         0.7636

Mean ± Std:
      val_f1_macro  test_f1_macro  test_bal_acc  test_accuracy
mean        0.7749         0.7488        0.7441         0.7674
std         0.0166         0.0250        0.0197         0.0166

--- Head-to-head on 5-way regional scrubbed+ ---
RoBERTa    (weighted, lr=2e-5): 0.7999 ± 0.0094
ModernBERT (plain,    lr=3e-5): 0.7488 ± 0.0250

Reference — country-level (07.1, 15 classes, 6820 rows):
  RoBERTa    0.6427 ± 0.0068
  ModernBERT 0.5903 ± 

## Save results

In [8]:
out = {
    'notebook': '12_Origin_Regional_5Way',
    'task': f'{len(label_names)}-way regional classification',
    'text_column': TEXT_COLUMN,
    'text_columns_used': EXTRA_TEXT_COLS,
    'scrub_tiers': ['countries_and_adjectivals', 'coffee_region_aliases', 'cultivars', 'producer_context_terms'],
    'num_scrub_terms': len(all_scrub_terms),
    'region_mapping': COUNTRY_TO_REGION,
    'n_rows': int(len(work)),
    'n_classes': int(work['origin_region'].nunique()),
    'region_distribution': work['origin_region'].value_counts().to_dict(),
    'country_distribution': work['origin_country'].value_counts().to_dict(),
    'post_scrub_country_leakage_rate': leak_rate,
    'post_scrub_region_token_leakage_rate': region_leak_rate,
    'roberta_seed_harness': {
        'model': ROBERTA_CKPT,
        'config': {'lr': ROBERTA_BEST_LR, 'weighted': ROBERTA_BEST_WEIGHTED, 'epochs': 12, 'early_stop_patience': 3},
        'seeds': SEEDS, 'runs': roberta_seed_results,
        'mean': roberta_df.drop(columns=['seed']).mean().to_dict(),
        'std':  roberta_df.drop(columns=['seed']).std().to_dict(),
    },
    'modernbert_seed_harness': {
        'model': MODERNBERT_CKPT,
        'config': {'lr': MODERNBERT_BEST_LR, 'weighted': MODERNBERT_BEST_WEIGHTED, 'epochs': 12, 'early_stop_patience': 3},
        'seeds': SEEDS, 'runs': modernbert_seed_results,
        'mean': modernbert_df.drop(columns=['seed']).mean().to_dict(),
        'std':  modernbert_df.drop(columns=['seed']).std().to_dict(),
    },
}
out_path = os.path.join(OUTPUT_DIR_ROOT, 'results.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, default=float)
print('Saved:', out_path)


Saved: artifacts/origin_region_5way_scrubbed_plus\results.json
